<a href="https://colab.research.google.com/github/askette828/TextMemorizer/blob/master/Copy_of_starter_housing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import polars as pl
housing = pl.read_csv('thttps://raw.githubusercontent.com/byui-cse/cse450-course/master/data/housing.csv')

In [ ]:
housing.head()

id,date,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,price
i64,str,i64,f64,i64,i64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,i64,i64,f64
1565930130,"""20141104T000000""",4,3.25,3760,4675,2.0,0,0,3,8,2740,1020,2007,0,98038,47.3862,-122.048,3280,4033,429900.0
3279000420,"""20150115T000000""",3,1.75,1460,7800,1.0,0,0,2,7,1040,420,1979,0,98023,47.3035,-122.382,1310,7865,233000.0
194000575,"""20141014T000000""",4,1.0,1340,5800,1.5,0,2,3,7,1340,0,1914,0,98116,47.5658,-122.389,1900,5800,455000.0
2115510160,"""20141208T000000""",3,1.75,1440,8050,1.0,0,0,3,8,1440,0,1985,0,98023,47.3187,-122.39,1790,7488,258950.0
7522500005,"""20140815T000000""",2,1.5,1780,4750,1.0,0,0,4,7,1080,700,1947,0,98117,47.6859,-122.395,1690,5962,555000.0


In [ ]:


housing = housing.with_columns([
    (pl.col("sqft_above") + pl.col("sqft_basement")).alias("sqft_total"),
    (pl.col("sqft_total") / pl.col("sqft_lot")).alias("living_lot_ratio"),
    (pl.col("bathrooms") / pl.col("bedrooms").clip(lower_bound=1)).alias("bathrooms_per_bedroom"),
    (pl.col("yr_renovated") > 0).cast(pl.Int8).alias("was_renovated")
])

In [ ]:
housing.columns

['id',
 'date',
 'bedrooms',
 'bathrooms',
 'sqft_living',
 'sqft_lot',
 'floors',
 'waterfront',
 'view',
 'condition',
 'grade',
 'sqft_above',
 'sqft_basement',
 'yr_built',
 'yr_renovated',
 'zipcode',
 'lat',
 'long',
 'sqft_living15',
 'sqft_lot15',
 'price',
 'sqft_total',
 'living_lot_ratio',
 'bathrooms_per_bedroom',
 'was_renovated']

In [ ]:
features = [
    'bedrooms',
    'bathrooms',
    'sqft_living',
    'sqft_lot',
    'floors',
    'waterfront',
    'view',
    'condition',
    'grade',
    'sqft_above',
    'sqft_basement',
    'yr_built',
    'yr_renovated',
    'zipcode',
    'lat',
    'long',
    'sqft_living15',
    'sqft_lot15',
    'was_renovated',
    'bathrooms_per_bedroom',
    'living_lot_ratio',
    'sqft_total'
]

target = 'price'

Model

In [ ]:

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:

X = housing.select(features)

y = housing.select(target)


X = X.to_numpy()

y = y.to_numpy().ravel()


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = GradientBoostingRegressor(
    n_estimators=500, #makes more trees
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

model.fit(X_train, y_train)


y_pred = model.predict(X_test)


print("RMSE:", mean_squared_error(y_test, y_pred) ** 0.5)

print("R²:", r2_score(y_test, y_pred))

"""
 param_grid = {
    "n_estimators": [300, 500, 800],
    "learning_rate": [0.02, 0.05, 0.1],
    "max_depth": [2, 3, 4],
    "min_samples_leaf": [3, 5, 10],
    "subsample": [0.8, 1.0],
    "loss": ["squared_error", "huber"]
}

search = RandomizedSearchCV(
    estimator=GradientBoostingRegressor(random_state=42),
    param_distributions=param_grid,
    n_iter=20,
    scoring="neg_root_mean_squared_error",
    cv=3,
    random_state=42,
    n_jobs=-1
)

search.fit(X_train, y_train)

model = search.best_estimator_
y_pred = model.predict(X_test)

print("Best parameters:")
print(search.best_params_)

Best parameters:
{'subsample': 0.8, 'n_estimators': 500, 'min_samples_leaf': 10, 'max_depth': 4, 'loss': 'squared_error', 'learning_rate': 0.1}
RMSE: 126751.39794509379
R²: 0.8934750630477881
"""

RMSE: 121670.29374551527
R²: 0.9018444449618866


'\n param_grid = {\n    "n_estimators": [300, 500, 800],\n    "learning_rate": [0.02, 0.05, 0.1],\n    "max_depth": [2, 3, 4],\n    "min_samples_leaf": [3, 5, 10],\n    "subsample": [0.8, 1.0],\n    "loss": ["squared_error", "huber"]\n}\n\nsearch = RandomizedSearchCV(\n    estimator=GradientBoostingRegressor(random_state=42),\n    param_distributions=param_grid,\n    n_iter=20,\n    scoring="neg_root_mean_squared_error",\n    cv=3,\n    random_state=42,\n    n_jobs=-1\n)\n\nsearch.fit(X_train, y_train)\n\nmodel = search.best_estimator_\ny_pred = model.predict(X_test)\n\nprint("Best parameters:")\nprint(search.best_params_)\n\nBest parameters:\n{\'subsample\': 0.8, \'n_estimators\': 500, \'min_samples_leaf\': 10, \'max_depth\': 4, \'loss\': \'squared_error\', \'learning_rate\': 0.1}\nRMSE: 126751.39794509379\nR²: 0.8934750630477881\n'

In [ ]:
importance = pl.DataFrame({
    "Feature": features,
    "Importance": model.feature_importances_
})

print(importance.sort("Importance", descending=True))

shape: (23, 2)
┌───────────────────────┬────────────┐
│ Feature               ┆ Importance │
│ ---                   ┆ ---        │
│ str                   ┆ f64        │
╞═══════════════════════╪════════════╡
│ grade                 ┆ 0.287448   │
│ sqft_total            ┆ 0.16888    │
│ lat                   ┆ 0.158903   │
│ sqft_living           ┆ 0.145977   │
│ long                  ┆ 0.064303   │
│ …                     ┆ …          │
│ bathrooms_per_bedroom ┆ 0.002355   │
│ floors                ┆ 0.000808   │
│ bedrooms              ┆ 0.000376   │
│ was_renovated         ┆ 0.000104   │
│ size_tier             ┆ 0.000009   │
└───────────────────────┴────────────┘


In [ ]:
mini=pl.read_csv("https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/housing_holdout_test_mini.csv")

In [ ]:
X_train= housing.select(features)
y_train= housing.select(target)
X_test= mini.select(features)

In [ ]:

model.fit(X_train, y_train)

predictions = model.predict(X_test)


results_df = pl.DataFrame({
    "price": predictions
})

results_df.write_csv("gb-module3-predictions.csv")
